In [1]:
%load_ext autoreload
%autoreload 2
import os
import json
import torch
import pycolmap
import numpy as np
import PIL.Image as Image
import matplotlib.pyplot as plt
import torchvision.transforms as T

from mylib.plot import plot_imgs

In [2]:
dataset = "mipnerf360" # terrasky3D, mipnerf360, scannetpp
scene = "bicycle" # vienna_state_opera, bicycle, bonsai, graz_townhall, graz_church, 40aec5fffa

# Load dataset paths and parameters from JSON
with open("benchmarks/paths.json") as f:
    paths_cfg = json.load(f)

dataset_cfg = paths_cfg[dataset]

images_path = os.path.join(
    dataset_cfg["images_path"], scene, dataset_cfg["images_folder"]
)
base_path = dataset_cfg["base_path"]
reconstruction_path = os.path.join(
    base_path, scene, dataset_cfg["reconstruction_folder"]
)
depths_path = os.path.join(
    base_path,
    scene,
    dataset_cfg.get("depths_folder", dataset_cfg.get("depth_folder", "")),
)
gt_path = os.path.join(dataset_cfg["gt_path"], scene, dataset_cfg["gt_folder"])
opt = f"optimized_reconstruction_GD/{scene}"

print("Images path:", images_path)
print("Reconstruction path:", reconstruction_path)
print("Depths path:", depths_path)
print("GT path:", gt_path)

Images path: /home/mattia/Desktop/datasets/mipnerf360/bicycle/images_150
Reconstruction path: /home/mattia/Desktop/Repos/batchsfm/benchmarks/vggt/mipnerf360/bicycle/sparse
Depths path: /home/mattia/Desktop/Repos/batchsfm/benchmarks/vggt/mipnerf360/bicycle/sparse/depth_maps
GT path: /home/mattia/Desktop/datasets/mipnerf360/bicycle/sparse_150


In [3]:
from adjuster import Adjuster

adjuster = Adjuster(
    reconstruction_path = reconstruction_path,
    images_path = images_path,
    depths_path = depths_path,
    k_lr=1e-3, grad_k=True,
    z_lr=3e-3, grad_z=True,
    use_mlp_pose_refinement=True,
    mlp_pose_lr=3e-3,
    detector="canny",  # or "canny", "bdcn", "sam2", "diff"
    detector_params={"low_threshold":0.20, "high_threshold":0.25, "kernel_size":7, "sigma":2},

    matcher_type="exhaustive",
    viz=True,
    verbose=True,
    max_edges_points=1024*12,
    max_viewgraph_pairs=1024*4,
    single_camera_per_folder=True,

    max_num_iterations=2000,
)

CannyEdgeDetector initialized with low_threshold=0.2, high_threshold=0.25, hysteresis=True, kernel_size=7, sigma=2, device=cuda
Found 150 images in /home/mattia/Desktop/datasets/mipnerf360/bicycle/images_150
When using MLP pose refinement, q and t gradients are disabled.
Edges stats:
 53 images have more than 12,288 edges. 
 max: 23,100 | min: 3,451 | avg: 10,747 | std: 4,325.23 | quantiles (0.5, 0.9): 9,668, 16,149
Filtered viewgraph: 2,474/11,175 pairs retained
Average degree: 32.99, min connection 6, less than 5 neighbors: 0 images

Total parameters to optimize:
  t:                0
  q:                0
  mlp:        269,580
  k:                2
  z:        3,686,400
-----------------------
  Total:    3,955,982



In [125]:
import matplotlib.pyplot as plt
import numpy as np
import torch
from itertools import combinations

# Parameters
window1 = 25
tol1 = 0.5

# early_stop_type="pose"
# tol2_list = [0.15, 0.1, 0.09, 0.08, 0.07]
# window2_list = [25, 50, 75, 100]

early_stop_type="loss"
tol2_list = [1e-3, 5e-4, 1e-4, 5e-5, 1e-5]
window2_list = [25, 50, 75, 100]

res = {}
for dataset in ["mipnerf360", "terrasky3D", "scannetpp"]:
    for scene in os.listdir(f"benchmarks/vggt_edge_canny_final_no_stop/{dataset}"):
        data = json.load(open(f"benchmarks/vggt_edge_canny_final_no_stop/{dataset}/{scene}/sparse/training_logs.json"))
        steps = np.arange(data["steps_actual"])
        max_changes = np.array([x.item() if torch.is_tensor(x) else x for x in data['list_changes']['max']])
        auc = data['list_auc']['auc']
        auc = {int(k):v for k,v in auc.items()}
        raw_loss = np.array(data['list_loss'])

        for tol2 in tol2_list:
            for window2 in window2_list:
                # Plot Smoothed Line for Window 1
                if len(max_changes) >= window1:
                    smoothed1 = np.convolve(max_changes, np.ones(window1)/window1, mode='valid')
                    # Offset steps to align with the end of the rolling window
                    # ax1.plot(steps[window1-1:], smoothed1, label=f'Smoothed (W={window1})', color='darkblue', linewidth=2, zorder=3)

                # Plot Smoothed Line for Window 2 (Visualizing what Tol2 sees)
                if len(max_changes) >= window2:
                    smoothed2 = np.convolve(max_changes, np.ones(window2)/window2, mode='valid')
                    # ax1.plot(steps[window2-1:], smoothed2, label=f'Smoothed (W={window2})', color='teal', linewidth=1.5, linestyle='--', zorder=3)

                # --- Convergence Detection ---
                # We calculate convergence at each step based on the preceding window
                conv1 = [adjuster.check_convergence(max_changes[:i+1], "pose", window1, tol1) for i in range(len(max_changes))]

                if early_stop_type == "pose":
                    conv2 = [adjuster.check_convergence(max_changes[:i+1], "pose", window2, tol2) for i in range(len(max_changes))]
                else:
                    conv2 = [adjuster.check_convergence(raw_loss[:i+1], "loss", window2, tol2) for i in range(len(raw_loss))]

                # just need to fined when it converges after conv 1
                # find step when conv1 became true
                for i, conv_step in enumerate(conv1):
                    conv_step = bool(conv_step)
                    if conv_step:
                        conv1_step = i
                        break

                for i, conv_step in enumerate(conv2):
                    conv_step = bool(conv_step)
                    if conv_step and i > conv1_step:
                        conv2_step = i
                        break


                # Scale loss for plotting (visual only) relative to max_changes range
                if len(raw_loss) > 0:
                    # Use steps[:len(raw_loss)] to ensure X-axis alignment if lengths differ
                    loss_x = steps[:len(raw_loss)] 
                    l_min, l_max = raw_loss.min(), raw_loss.max()
                    m_min, m_max = max_changes.min(), max_changes.max()
                    
                    # Avoid division by zero if loss is constant
                    denom = (l_max - l_min) if l_max != l_min else 1
                    scaled_loss = (raw_loss - l_min) / denom * (m_max - m_min) + m_min
                    

                auc_score = auc[5][conv2_step//50]

                res[(scene, tol2, window2)] = (conv1_step, conv2_step, round(auc_score,1))

In [126]:
import pandas as pd
df = pd.DataFrame.from_dict(res, orient='index', columns=['Conv1 Step', 'Conv2 Step', 'AUC Score'])
# split index in columns
df[['Scene', 'Tol2', 'Window2']] = pd.DataFrame(df.index.tolist(), index=df.index)
df = df.reset_index(drop=True)

# group by scene, sort by auc, pick highest
df_  = df.groupby('Scene').apply(lambda x: x.loc[x['AUC Score'].idxmax()])
df_ 

/tmp/ipykernel_67730/1590722150.py:8: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  df_  = df.groupby('Scene').apply(lambda x: x.loc[x['AUC Score'].idxmax()])


,Conv1 Step,Conv2 Step,AUC Score,Scene,Tol2,Window2
Scene,,,,,,
09c1414f1b,109,289,59.6,09c1414f1b,0.00100,50
1ada7a0617,63,166,68.1,1ada7a0617,0.00100,25
21d970d8de,89,233,84.2,21d970d8de,0.00100,50
286b55a2bf,95,251,74.3,286b55a2bf,0.00100,50
38d58a7a31,68,183,78.2,38d58a7a31,0.00100,25
3e8bba0176,52,278,83.5,3e8bba0176,0.00100,100
40aec5fffa,62,1134,69.3,40aec5fffa,0.00010,25
578511c8a9,396,1039,55.9,578511c8a9,0.00010,25
5f99900f09,56,841,76.7,5f99900f09,0.00010,50


In [155]:
# Extract scalar values from mode() results
tol2 = 1e-4 #df_.Tol2.mode()[0]
window2 = 100 #df_.Window2.mode()[0]

print("early_stop_type:", early_stop_type)
print("tol2:", tol2)
print("window2:", window2)

# group by scene first, then select row with mode values
df_selected = df.groupby('Scene',).apply(lambda x: x[(x['Tol2'] == tol2) & (x['Window2'] == window2)], include_groups=False)
df_selected.loc['mean'] = df_selected.mean(numeric_only=True)
df_selected

early_stop_type: loss
tol2: 0.0001
window2: 100


,Conv1 Step,Conv2 Step,AUC Score,Tol2,Window2
"(09c1414f1b, 711)",109.000000,855.000000,54.000000,0.0001,100.0
"(1ada7a0617, 471)",63.000000,354.000000,66.400000,0.0001,100.0
"(21d970d8de, 591)",89.000000,461.000000,83.000000,0.0001,100.0
"(286b55a2bf, 611)",95.000000,659.000000,69.500000,0.0001,100.0
"(38d58a7a31, 351)",68.000000,556.000000,76.900000,0.0001,100.0
"(3e8bba0176, 431)",52.000000,406.000000,83.300000,0.0001,100.0
"(40aec5fffa, 491)",62.000000,1010.000000,69.300000,0.0001,100.0
"(578511c8a9, 631)",396.000000,986.000000,55.800000,0.0001,100.0
"(5f99900f09, 551)",56.000000,888.000000,76.600000,0.0001,100.0
"(7831862f02, 371)",50.000000,465.000000,85.200000,0.0001,100.0


In [159]:
for tol2 in tol2_list:
    for window2 in window2_list:
        df_selected = df.groupby('Scene',).apply(lambda x: x[(x['Tol2'] == tol2) & (x['Window2'] == window2)], include_groups=False)
        df_selected.loc['mean'] = df_selected.mean(numeric_only=True)
        print("tol2:", tol2,"window2:", window2, "AUC Score:", df_selected.loc['mean']["AUC Score"], "steps:", df_selected.loc['mean']["Conv2 Step"])

tol2: 0.001 window2: 25 AUC Score: 74.15000000000002 steps: 226.05555555555554
tol2: 0.001 window2: 50 AUC Score: 75.4638888888889 steps: 275.80555555555554
tol2: 0.001 window2: 75 AUC Score: 75.69722222222222 steps: 311.22222222222223
tol2: 0.001 window2: 100 AUC Score: 75.84999999999998 steps: 348.1111111111111
tol2: 0.0005 window2: 25 AUC Score: 75.64166666666665 steps: 306.02777777777777
tol2: 0.0005 window2: 50 AUC Score: 75.85833333333332 steps: 315.55555555555554
tol2: 0.0005 window2: 75 AUC Score: 75.93888888888888 steps: 345.1388888888889
tol2: 0.0005 window2: 100 AUC Score: 76.04166666666667 steps: 373.0
tol2: 0.0001 window2: 25 AUC Score: 76.68055555555557 steps: 714.25
tol2: 0.0001 window2: 50 AUC Score: 76.25555555555556 steps: 642.3055555555555
tol2: 0.0001 window2: 75 AUC Score: 76.81666666666666 steps: 647.3611111111111
tol2: 0.0001 window2: 100 AUC Score: 76.81388888888888 steps: 663.8611111111111
tol2: 5e-05 window2: 25 AUC Score: 76.76388888888889 steps: 942.69444444

In [128]:
# with pose
# mip 0.07, 75
# terrasky3D 0.1, 100
# scannetpp 0.15, 25

In [129]:
# with loss
# mipnerf360: 5e-5, 100
# terrasky3d: 5e-5, 15
# scannetpp: 0.001, 15